In [1]:
!sudo apt-get update -qq
!sudo apt-get install -y -qq zstd lshw
!curl -fsSL https://ollama.com/install.sh | sh

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 4.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package lshw.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../lshw_02.19.git.2021.06.19.996aaad9c7-2ubuntu0.24.04.1_amd64.deb ...
Unpacking lshw (02.19.git.2021.06.19.996aaad9c7-2ubuntu0.24.04.1) ...
Selecting previously unselected package pci.ids.
Preparing to unpack .../pci

In [2]:
%%bash --bg
ollama serve > ollama.log 2>&1

In [3]:
!sleep 5
!ollama pull llama3.2
!ollama list


NAME               ID              SIZE      MODIFIED               
llama3.2:latest    a80c4f17acd5    2.0 GB    Less than a second ago    


In [4]:
%pip install -q langgraph-supervisor langchain langchain-ollama

In [ ]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="llama3.2:latest", temperature=0, num_ctx=8192)

print(model.invoke("Say hello to my students in one short sentence.").content)

In [ ]:
#simple data
CURRICULUM_DB = {
    "Q1": {
        "question": "What is overfitting in machine learning?",
        "model_answer": (
            "Overfitting happens when a model learns noise and specific details in the "
            "training data too closely, hurting generalization on unseen data."
        ),
        "key_concepts": ["training data noise", "poor generalization", "unseen/test data"],
        "max_score": 10,
    },
    "Q2": {
        "question": "What is the difference between precision and recall?",
        "model_answer": (
            "Precision measures how many predicted positives are actually positive, "
            "while recall measures how many actual positives were correctly identified."
        ),
        "key_concepts": ["predicted positives", "actual positives", "false positives vs false negatives"],
        "max_score": 10,
    },
}

In [ ]:
def Matching_questions(student_query: str) -> str:
    """Match a student question/topic to a reference question in the curriculum."""
    q = student_query.lower()
    if "overfit" in q or "training" in q or "generaliz" in q:
        item = CURRICULUM_DB["Q1"]
        return (
            f"Matched ID: Q1 | Confidence: 0.95\n"
            f"Canonical Question: {item['question']}\n"
            f"Key Concepts: {', '.join(item['key_concepts'])}\n"
            f"Model Answer: {item['model_answer']}"
        )
    if "precision" in q or "recall" in q:
        item = CURRICULUM_DB["Q2"]
        return (
            f"Matched ID: Q2 | Confidence: 0.94\n"
            f"Canonical Question: {item['question']}\n"
            f"Key Concepts: {', '.join(item['key_concepts'])}\n"
            f"Model Answer: {item['model_answer']}"
        )
    return "Matched ID: NONE | Confidence: 0.15 | No clear match found. Ask the student to clarify."

In [ ]:
def Comparison_answers(question_id: str, student_answer: str) -> str:
    """Compare student_answer against the matched question's model answer and key concepts."""
    item = CURRICULUM_DB.get(question_id)
    if not item:
        return "Error: Invalid question_id. Match the question first."

    ans = student_answer.lower()
    got_right = []
    missing = []

    if "noise" in ans or "memoriz" in ans or "too closely" in ans or "training" in ans:
        got_right.append("Mentions fitting training data/noise too closely.")
    else:
        missing.append("Explain how the model fits training data/noise too closely.")

    if "unseen" in ans or "test" in ans or "generaliz" in ans or "new data" in ans:
        got_right.append("Connects the issue to generalization on unseen/test data.")
    else:
        missing.append("State that it hurts generalization on unseen/test data.")

    if not got_right:
        got_right.append("Attempted to answer the question.")

    return (
        f"Question ID: {question_id}\n"
        f"Correct Points: {got_right}\n"
        f"Missing / Improvement Areas: {missing}\n"
        f"Reference Model Answer: {item['model_answer']}"
    )



In [ ]:

def answer_score(question_id: str, correct_count: int, missing_count: int) -> str:
    """Calculate the official score based on comparison counts."""
    item = CURRICULUM_DB.get(question_id)
    if not item:
        return "Score: 0/10 (Invalid question_id)"

    max_score = item["max_score"]
    total_points = max(1, correct_count + missing_count)
    score = round((correct_count / total_points) * max_score, 1)
    return f"Official Score for {question_id}: {score} / {max_score}"

In [ ]:
scorer_agent = create_agent(
    model,
    tools=[answer_score],
    name="answer_scorer",
    system_prompt=(
        "You are the Scoring sub-agent. "
        "Always call answer_score to get the official score. Never invent a score on your own."
    ),
)

NameError: name 'create_agent' is not defined

In [ ]:
supervisor_prompt = (
    "You are the Teacher Supervisor managing three sub-agents: "
    "question_matcher, answer_comparator, and answer_scorer.\n"
    "Sub-agents never talk to each other; you coordinate every step:\n"
    "1. First, transfer to question_matcher to find the question_id and model answer.\n"
    "   - If confidence is low or no match is found, ask the student to clarify without grading.\n"
    "2. Second, transfer to answer_comparator with the question_id and student's answer.\n"
    "3. Third, transfer to answer_scorer with the question_id and counts of correct/missing points.\n"
    "4. Finally, write the response in the same language as the student using ONLY these 4 sections:\n"
    "   - Score\n"
    "   - What You Got Right\n"
    "   - Areas for Improvement\n"
    "   - Model Answer / Tip"
)

teacher_supervisor = create_supervisor(
    [matcher_agent, comparator_agent, scorer_agent],
    model=model,
    prompt=supervisor_prompt,
    include_agent_name="inline",
    output_mode="full_history",
)

app = teacher_supervisor.compile()

In [ ]:
sample_submission = (
    "Question: What is overfitting?\n"
    "Student Answer: The model memorizes the training data too much, "
    "including noise, and performs poorly on new data."
)

result = app.invoke({"messages": [{"role": "user", "content": sample_submission}]})

for message in result["messages"]:
    message.pretty_print()

In [ ]:
memory_app = teacher_supervisor.compile(checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "teacher-session-1"}}

turn1 = memory_app.invoke(
    {"messages": [{"role": "user", "content": sample_submission}]},
    config,
)
print("Turn 1 Final:", turn1["messages"][-1].content)

turn2 = memory_app.invoke(
    {"messages": [{"role": "user", "content": "Can you re-evaluate if I also say it fails on unseen test data?"}]},
    config,
)
print("Turn 2 Final:", turn2["messages"][-1].content)

for namespace, update in app.stream(
    {"messages": [{"role": "user", "content": sample_submission}]},
    stream_mode="updates",
    subgraphs=True,
):
    print(namespace, "→", list(update.keys()))